In [1]:
!pip install -q sentence-transformers tiktoken nltk transformers accelerate bitsandbytes

In [2]:
import os

folder_path = "/home/jupyter/cooked-cockatoo/bh-nlp/nlp/src/documents"  # update if needed

corpus = {}
for i, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.endswith(".txt"):
        filepath = os.path.join(folder_path, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            corpus[f"doc_{i}"] = f.read().strip()

print(f"Loaded {len(corpus)} documents")

Loaded 296 documents


In [3]:
import nltk
import tiktoken
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

enc = tiktoken.get_encoding("cl100k_base")

def semantic_chunk(text, max_tokens=400, overlap_sentences=2):
    sentences = sent_tokenize(text)
    chunks = []
    current_sentences = []
    current_token_count = 0
    for sentence in sentences:
        sentence_tokens = len(enc.encode(sentence))
        if current_token_count + sentence_tokens > max_tokens and current_sentences:
            chunks.append(" ".join(current_sentences))
            current_sentences = current_sentences[-overlap_sentences:]
            current_token_count = sum(len(enc.encode(s)) for s in current_sentences)
        current_sentences.append(sentence)
        current_token_count += sentence_tokens
    if current_sentences:
        chunks.append(" ".join(current_sentences))
    return chunks

all_chunks = []
for doc_id, text in corpus.items():
    for i, chunk_text in enumerate(semantic_chunk(text)):
        all_chunks.append({"source": doc_id, "chunk_id": i, "text": chunk_text})

print(f"Total chunks: {len(all_chunks)}")
print(f"Sample chunk:\n{all_chunks[0]['text'][:200]}")

Total chunks: 1315
Sample chunk:
# The Dissolution of Wampa Robotics: A Structural Analysis of Coordinated Corporate Action Within the CGC Framework

**Haven Institute for Policy Studies | Research Paper**
**Dr. Elara Quinshan**
**74


In [4]:
import math
from collections import Counter

class BM25:
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        self.k1 = k1
        self.b = b
        self.N = len(chunks)
        self.tokenized = [self._tokenize(c["text"]) for c in chunks]
        self.avgdl = sum(len(d) for d in self.tokenized) / self.N
        self.df = {}
        for doc in self.tokenized:
            for term in set(doc):
                self.df[term] = self.df.get(term, 0) + 1

    def _tokenize(self, text):
        return text.lower().split()

    def _score(self, query_terms, idx):
        doc_tokens = self.tokenized[idx]
        doc_len = len(doc_tokens)
        tf = Counter(doc_tokens)
        score = 0.0
        for term in query_terms:
            if term not in self.df:
                continue
            idf = math.log((self.N - self.df[term] + 0.5) / (self.df[term] + 0.5) + 1)
            tf_norm = (tf[term] * (self.k1 + 1)) / (tf[term] + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl))
            score += idf * tf_norm
        return score

    def search(self, query, top_k=5):
        query_terms = self._tokenize(query)
        scores = [self._score(query_terms, i) for i in range(self.N)]
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [{"source": self.chunks[i]["source"], "chunk_id": self.chunks[i]["chunk_id"],
                 "score": scores[i], "text": self.chunks[i]["text"]} for i in top_indices]

bm25 = BM25(all_chunks)
print(f"BM25 built on {len(all_chunks)} chunks")

BM25 built on 1315 chunks


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

class DenseRetriever:
    def __init__(self, chunks, model_name="BAAI/bge-small-en-v1.5"):
        self.chunks = chunks
        self.model = SentenceTransformer(model_name)
        print("Embedding chunks...")
        texts = [c["text"] for c in chunks]
        self.embeddings = self.model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
        print("Done!")

    def search(self, query, top_k=5):
        query_embedding = self.model.encode(query, normalize_embeddings=True)
        scores = self.embeddings @ query_embedding
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [{"source": self.chunks[i]["source"], "chunk_id": self.chunks[i]["chunk_id"],
                 "score": float(scores[i]), "text": self.chunks[i]["text"]} for i in top_indices]

dense = DenseRetriever(all_chunks)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding chunks...


Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Done!


In [6]:
class RRF:
    def __init__(self, bm25, dense, k=60):
        self.bm25 = bm25
        self.dense = dense
        self.k = k

    def search(self, query, top_k=5, bm25_candidates=20, dense_candidates=20):
        bm25_results = self.bm25.search(query, top_k=bm25_candidates)
        dense_results = self.dense.search(query, top_k=dense_candidates)
        rrf_scores = {}
        chunk_map = {}
        for rank, r in enumerate(bm25_results):
            key = (r["source"], r["chunk_id"])
            rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (self.k + rank + 1)
            chunk_map[key] = r
        for rank, r in enumerate(dense_results):
            key = (r["source"], r["chunk_id"])
            rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (self.k + rank + 1)
            chunk_map[key] = r
        ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return [{"source": key[0], "chunk_id": key[1], "score": score,
                 "text": chunk_map[key]["text"]} for key, score in ranked]

rrf = RRF(bm25, dense)
print("RRF ready")

RRF ready


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print(f"GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Model device: {next(model.parameters()).device}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

GPU allocated: 1.36 GB
Model device: cuda:0


In [9]:
def get_expanded_chunks(rrf_results, all_chunks, window=1):
    """Retrieve top RRF chunks plus their neighbours for context continuity."""
    chunk_lookup = {(c["source"], c["chunk_id"]): i for i, c in enumerate(all_chunks)}
    seen = set()
    expanded = []
    for r in rrf_results:
        for offset in range(-window, window + 1):
            neighbour_id = r["chunk_id"] + offset
            key = (r["source"], neighbour_id)
            if key in chunk_lookup and key not in seen:
                seen.add(key)
                expanded.append(all_chunks[chunk_lookup[key]])
    return expanded


def generate_answer(question, retrieved_docs, max_new_tokens=200):
    """Generate a grounded answer using the LLM."""
    context = "\n\n".join([f"[{r['source']}]: {r['text']}" for r in retrieved_docs])
    prompt = (
        "You are a precise question-answering assistant. Answer the question using ONLY the provided context.\n"
        "- Use the passage that most directly answers the question\n"
        "- Answer concisely and directly in 1-2 sentences\n"
        "- For time difference questions, subtract the two dates and stop\n"
        "- If the answer is not in the context, say 'I don't have enough information'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)


def ask(question, top_k=8, verbose=True):
    """Full RAG pipeline: question → retrieve → generate → answer."""
    rrf_results = rrf.search(question, top_k=top_k)
    retrieved_docs = get_expanded_chunks(rrf_results, all_chunks, window=1)[:5]
    if verbose:
        print("Retrieved sources:")
        for r in retrieved_docs:
            print(f"  • {r['source']} (chunk {r['chunk_id']})")
        print()
    return generate_answer(question, retrieved_docs)


print("Pipeline ready — use ask('your question') to query your corpus")

Pipeline ready — use ask('your question') to query your corpus


In [10]:
import re

def normalize(text):
    text = re.sub(r"[$£€¥]", "", text)
    return re.sub(r"[^\w\s]", "", text.lower()).strip()

def expand_numbers(text):
    text = re.sub(r"(\d),(\d{3})(?=\D|$)", r"\1\2", text)
    text = re.sub(r"(\d),(\d{3})(?=\D|$)", r"\1\2", text)
    def replace_million(m):
        val = float(m.group(1)) * 1_000_000
        return str(int(val) if val == int(val) else val)
    def replace_billion(m):
        val = float(m.group(1)) * 1_000_000_000
        return str(int(val) if val == int(val) else val)
    text = re.sub(r"([\d.]+)\s*million", replace_million, text, flags=re.IGNORECASE)
    text = re.sub(r"([\d.]+)\s*billion", replace_billion, text, flags=re.IGNORECASE)
    return text

def key_terms_match(expected, predicted):
    exp = normalize(expand_numbers(expected))
    pred = normalize(expand_numbers(predicted))
    if exp in pred:
        return True
    if "less than one year" in expected.lower():
        return len(re.findall(r"0\.\d+", predicted)) > 0
    stopwords = {"and", "the", "a", "an", "of", "to", "in", "is", "was", "were",
                 "by", "at", "over", "than", "approximately", "about", "roughly"}
    key_terms = [t for t in exp.split() if t not in stopwords]
    if not key_terms:
        return False
    def term_in_pred(term):
        if term in pred: return True
        if term.endswith("s") and term[:-1] in pred: return True
        if not term.endswith("s") and term + "s" in pred: return True
        return False
    matched = sum(1 for t in key_terms if term_in_pred(t))
    return matched / len(key_terms) >= 0.75


eval_set = [
    {"question": "What penalty was assessed against Halcyon Technical Services in CGC enforcement action EA-76-088?", "answer": "4.5 million Credits and mandatory equipment surrender"},
    {"question": "What internal codename did ONE Network Enterprises use for its classified data-sharing arrangement with the CGC Security Directorate?", "answer": "SEASTITCH"},
    {"question": "Given the projected annual revenue and the total development cost for Project Liminal, how many years of post-launch recurring revenue at the low end of projections would be needed to recoup the initial investment?", "answer": "less than one year"},
    {"question": "By what deadline were Phase II vessel deliveries required to be completed under Nguyen Anh Mai's 77 PCE fleet modernization directive?", "answer": "Q4 78 PCE"},
    {"question": "How many years passed between Cyanite Industries being mandated as sole Floodwall contractor under the Blackshore Accords and the completion of the Phase III nuclear transition that the 2119 anniversary article centers on?", "answer": "approximately 37 years"},
    {"question": "What industry did Park Soo-Hyun come from before becoming CEO of Renhwa Media?", "answer": "Sharpsea Bloc logistics"},
    {"question": "What share of Haven's economic transactions are conducted through Edge-integrated payment interfaces?", "answer": "approximately 71%"},
    {"question": "How large was the port modernization program announced by Chairperson Idako at the Cape Tidak Trade Forum in early 2118?", "answer": "340 million Phi Credits"},
    {"question": "Which team won the South Haven Waterfront League championship in 76 PCE, and by what score?", "answer": "Dockside Currents, 47 points to 39 over the Floodwall Runners"},
]


def evaluate(eval_set, top_k=8):
    results = []
    for i, item in enumerate(eval_set):
        print(f"[{i+1}/{len(eval_set)}] {item['question']}")
        rrf_results = rrf.search(item["question"], top_k=top_k)
        retrieved_docs = get_expanded_chunks(rrf_results, all_chunks, window=1)[:5]
        predicted = generate_answer(item["question"], retrieved_docs)
        match = key_terms_match(item["answer"], predicted)
        results.append({"question": item["question"], "expected": item["answer"],
                        "predicted": predicted, "match": match})
        print(f"  Expected:  {item['answer']}")
        print(f"  Predicted: {predicted}")
        print(f"  {'✅' if match else '❌'}\n")
    total = len(results)
    passed = sum(r["match"] for r in results)
    print("=" * 60)
    print(f"Score: {passed}/{total} ({passed/total*100:.1f}%)")
    return results


results = evaluate(eval_set)

[1/9] What penalty was assessed against Halcyon Technical Services in CGC enforcement action EA-76-088?


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  Expected:  4.5 million Credits and mandatory equipment surrender
  Predicted: The financial penalty assessed against Halcyon Technical Services is 4,500,000 Phi Credits (four million five hundred thousand Credits), payable to the CGC Regulatory Compliance Fund within sixty (60) days of this filing's service date.
  ❌

[2/9] What internal codename did ONE Network Enterprises use for its classified data-sharing arrangement with the CGC Security Directorate?
  Expected:  SEASTITCH
  Predicted: SEASTITCH
  ✅

[3/9] Given the projected annual revenue and the total development cost for Project Liminal, how many years of post-launch recurring revenue at the low end of projections would be needed to recoup the initial investment?
  Expected:  less than one year
  Predicted: To determine how many years of post-launch recurring revenue at the low end of projections would be needed to recoup the initial investment, we need to calculate the required revenue based on the high-end estimates of the

In [ ]:
print("💬 Ask questions about your corpus. Type 'exit' to quit.\n")
print("-" * 50)

while True:
    question = input("\nYour question: ").strip()
    if question.lower() in ("exit", "quit", "q"):
        print("Bye!")
        break
    if not question:
        continue
    answer = ask(question)
    print(f"Answer: {answer}")
    print("-" * 50)

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import nltk
import tiktoken

# Download LLM
AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    cache_dir="src/models/qwen2.5-1.5b"
)
AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    cache_dir="src/models/qwen2.5-1.5b"
)

# Download embedding model
SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    cache_folder="src/models/bge-small"
)

# Download NLTK data
nltk.download("punkt", download_dir="src/models/nltk")
nltk.download("punkt_tab", download_dir="src/models/nltk")

# Cache tiktoken encoding
tiktoken.get_encoding("cl100k_base")

print("All models downloaded!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[nltk_data] Downloading package punkt to src/models/nltk...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/nltk/downloader.py:2395: RuntimeWarning: Security Warning [pathsec.ZipFile]: Path /home/jupyter/cooked-cockatoo/bh-nlp/nlp/src/src/models/nltk/tokenizers/punkt.zip allowed via CWD.
  zf = ZipFile(filename)


All models downloaded!


[nltk_data] Downloading package punkt_tab to src/models/nltk...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/nltk/downloader.py:2395: RuntimeWarning: Security Warning [pathsec.ZipFile]: Path /home/jupyter/cooked-cockatoo/bh-nlp/nlp/src/src/models/nltk/tokenizers/punkt_tab.zip allowed via CWD.
  zf = ZipFile(filename)
